# Notebook 00 — SPINK7-KLK5 Structure Acquisition and Docking

This notebook orchestrates the acquisition of the two protein structures required
for the SPINK7-KLK5 binding free energy pipeline:

1. **KLK5** (Kallikrein-Related Peptidase 5): Experimental crystal structure from the
   RCSB Protein Data Bank (PDB ID: [2PSX](https://www.rcsb.org/structure/2PSX)).
2. **SPINK7** (Serine Peptidase Inhibitor, Kazal Type 7): Predicted structure from the
   AlphaFold Protein Structure Database (UniProt accession: P58062).

After retrieving both monomers, a docked SPINK7-KLK5 complex is generated by
superposition onto the homologous SPINK5 (LEKTI domain)–KLK5 complex
(PDB: 4K1E) or via rigid-body docking (HADDOCK / ClusPro).

## Structural Biology Context

SPINK7 adopts the conserved Kazal-domain fold stabilized by three disulfide bonds.
Its reactive site loop (RSL) inserts into the KLK5 active-site cleft, positioning
the P1 residue into the S1 specificity pocket. The resulting complex buries
$\sim 800$–$1200~\text{\AA}^2$ of solvent-accessible surface area.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path for src imports.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR
from src.prep.pdb_fetch import fetch_pdb, fetch_alphafold

RAW_DIR = DATA_DIR / "pdb" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raw PDB directory: {RAW_DIR}")

## Step 1 — Retrieve KLK5 Crystal Structure (PDB: 2PSX)

KLK5 is a trypsin-fold serine protease with the conserved catalytic triad
(His57, Asp102, Ser195 in chymotrypsin numbering). The S1 pocket favors
basic residues (Arg/Lys) at the P1 position.

If `2PSX.pdb` already exists in the raw directory, the download is skipped.

In [ ]:
klk5_path = RAW_DIR / "2PSX.pdb"

if klk5_path.exists():
    print(f"KLK5 structure already present: {klk5_path}")
else:
    klk5_path = fetch_pdb("2PSX", RAW_DIR)
    print(f"Downloaded KLK5 structure to: {klk5_path}")

# Verify atom count.
import mdtraj as md

klk5 = md.load(str(klk5_path))
print(f"KLK5: {klk5.n_atoms} atoms, {klk5.n_residues} residues, "
      f"{klk5.n_chains} chain(s)")

## Step 2 — Retrieve SPINK7 AlphaFold2 Prediction (UniProt: P58062)

No experimental structure of SPINK7 has been deposited in the PDB as of the
pipeline development date. The AlphaFold2 predicted monomer (UniProt P58062)
is used as the starting structure.

> **Assumption Declaration:** The AlphaFold2 model provides a reliable fold
> prediction for the Kazal domain, but loop conformations — especially the RSL —
> must be validated by comparing predicted interface contacts with known
> Kazal-protease binding modes after docking.

In [ ]:
spink7_path = RAW_DIR / "SPINK7_AF2.pdb"

if spink7_path.exists():
    print(f"SPINK7 structure already present: {spink7_path}")
else:
    spink7_path = fetch_alphafold("P58062", RAW_DIR)
    print(f"Downloaded SPINK7 AlphaFold2 model to: {spink7_path}")

spink7 = md.load(str(spink7_path))
print(f"SPINK7: {spink7.n_atoms} atoms, {spink7.n_residues} residues, "
      f"{spink7.n_chains} chain(s)")

## Step 3 — Generate Docked SPINK7-KLK5 Complex

The docked complex can be generated via one of three approaches:

1. **Homology-based superposition** onto the SPINK5–KLK5 co-crystal (PDB: 4K1E)
   using backbone alignment of the catalytic triad region.
2. **HADDOCK web server** — define active residues from the RSL of SPINK7 and the
   S1 pocket of KLK5.
3. **ClusPro server** — upload SPINK7 and KLK5 as receptor/ligand for automated
   rigid-body docking.

### Validation Criteria

The best-scoring docked complex must satisfy:
- RSL of SPINK7 is positioned in the KLK5 active-site cleft.
- P1 residue is oriented toward the S1 pocket.
- No steric clashes (all pairwise $C_\alpha$ distances $> 2.0~\text{\AA}$).
- The catalytic triad (His57, Asp102, Ser195) is accessible.

**Note:** The docking step requires external tools and is performed outside this
notebook. Place the result at `data/pdb/raw/SPINK7_KLK5_docked.pdb`.

In [ ]:
docked_path = RAW_DIR / "SPINK7_KLK5_docked.pdb"

if docked_path.exists():
    complex_traj = md.load(str(docked_path))
    chains = list(complex_traj.topology.chains)
    print(f"Docked complex loaded: {complex_traj.n_atoms} atoms, "
          f"{len(chains)} chains")

    # Validate: at least 2 chains.
    assert len(chains) >= 2, (
        "Complex must contain at least two chains (SPINK7 + KLK5)."
    )

    # Check for steric clashes between chains.
    ca_indices_A = complex_traj.topology.select("chainid 0 and name CA")
    ca_indices_B = complex_traj.topology.select("chainid 1 and name CA")
    pos = complex_traj.xyz[0]  # nm
    min_dist_nm = float("inf")
    for i in ca_indices_A:
        for j in ca_indices_B:
            d = float(((pos[i] - pos[j]) ** 2).sum() ** 0.5)
            if d < min_dist_nm:
                min_dist_nm = d
    min_dist_angstrom = min_dist_nm * 10.0
    print(f"Minimum inter-chain CA distance: {min_dist_angstrom:.2f} A")
    assert min_dist_angstrom > 2.0, (
        f"Steric clash detected: minimum CA distance = {min_dist_angstrom:.2f} A"
    )
    print("Docked complex validation PASSED.")
else:
    print(f"Docked complex not yet available at {docked_path}.")
    print("Generate via HADDOCK, ClusPro, or homology superposition, ")
    print("then re-run this cell.")

## Next Steps

Once the docked complex is available at `data/pdb/raw/SPINK7_KLK5_docked.pdb`,
proceed to **Notebook 01 — System Preparation** to clean, protonate, solvate,
and parameterize the system for MD simulation:

```bash
python scripts/run_prep.py \
    --input-pdb data/pdb/raw/SPINK7_KLK5_docked.pdb \
    --chains A B \
    --ph 7.4 \
    --box-padding-nm 1.2 \
    --ionic-strength-molar 0.15
```